# Sod Shock Tube (2D)

The same Riemann problem as `sod_1d.ipynb`, extruded into a periodic slab.

Side | $p$ | $\rho$ | $v$
---|---|---|---
Left | 1.0 | 1.0 | 0.0
Right | 0.1795 | 0.25 | 0.0

**What is 2D about it.** Only the sampling, really -- the solution is still
one-dimensional, which is the point: any structure that appears along $y$ is
error, and it is visible directly in the plots below.

* **$x$** is arranged exactly as in 1D: periodic on $[-1, 1]$, the dense
  state occupying the middle half $|x| \le 0.5$ and the light state the two
  outer quarters. Nothing is reflected explicitly -- the mirror symmetry of
  that arrangement makes $x = 0$ and $x = \pm 1$ behave as reflecting walls
  until a wave reaches them, which is what makes a periodic box a shock tube.
  The analytic solution therefore describes the window $x \in [0, 1]$, which
  is what the profile panels draw.
* **$y$** is a plain periodic direction: a slab, whose width is measured in
  particle spacings rather than as a length (see the parameters cell).

**Equal mass, not equal spacing.** Sampling both states on the same lattice
would make the dense side's particles $\rho_l/\rho_r = 4$ times heavier, and
a 4:1 mass jump across the contact discontinuity is exactly where SPH's
density estimate misbehaves. So the light side is sampled $\sqrt{4} = 2$
times coarser in *both* directions instead, which equalises
$m = (\text{cell volume}) \times \rho$. In 1D that fell out of
`samplingRatio=4` being the density ratio; in 2D it takes a little arithmetic,
which the sampling cell below prints and lets you turn off (`equalMass`).

**How to read the profile panels.** Every particle is scattered against its
own $x$, with no averaging over $y$ -- the domain is periodic in $y$ and the
solution does not depend on it, so all of them are directly comparable to the
same 1D reference curve. The *vertical spread* at a given $x$ is then a real
measurement rather than a plotting artefact: it is the symmetry breaking, and
it shows up first at the shock and the contact. The last cell draws the same
field in the slab itself, where that structure is spatial rather than
statistical.

Like `sod_1d.ipynb`, this notebook is meant to be **edited while it runs**:
the step loop stays unrolled in a cell rather than hidden inside
`warpSPH.runner.run()`. Plotting calls `plotSod`/`plotSod_` directly rather
than going through `sod2dCase.setupPlot` -- see `sod_1d.ipynb`'s intro for why
that path does not live-update inside a Jupyter cell.

Precision note: switching between single and double precision is controlled
in the import cell below, and requires a kernel restart to take effect.

![](outputs/01-Sod_Shock_Tube_2D.gif)

In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.sod import states
from warpSPH.cases.sodND import sod2dCase
from warpSPH.caseUtils import plotSod, plotSod_, sodSampling, sodSamplingReport
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.runner.display import figureOf, visualizeWithFallback
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import dataclasses
import os
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `sod_2d.py`, made
# explicit and editable here. `sod2dCase.defaults`/`.params` are the same
# values the CLI script starts from -- anything not overridden below just keeps
# its case default.
spec = CaseSpec(caseName=sod2dCase.name, scheme=sod2dCase.scheme,
                params=dict(sod2dCase.params)).merged(**sod2dCase.defaults)

spec = spec.merged(
    # --- discretisation --------------------------------------------------
    nx=1000,                    # dense-side particles across its own half of the domain
    dim=2,
    L=2.0,

    # --- time stepping -----------------------------------------------------
    tLimit=0.15,
    dt=1e-3, adaptiveDt=True, cflFactor=0.3,

    # --- scheme --------------------------------------------------------
    kernel='B7',

    # --- output ------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=True,
    storeMode='trajectory',    # one growing trajectory.h5 (see sod_resume.ipynb)
    exportInterval=0.005,      # simulated-time interval between stored frames

    params=dict(
        gamma=5 / 3,
        left_rho=1.0, left_pressure=1.0, left_velocity=0.0,
        right_rho=0.25, right_pressure=0.1795, right_velocity=0.0,
        # The slab's width, in dense-side particle spacings. Measured that way
        # rather than as a length because the constraint it has to satisfy is a
        # multiple of the spacing: a slab narrower than twice the support radius
        # lets a particle interact with its own periodic image, silently. Fixing
        # it in spacings keeps that true at every `nx` -- the sampler checks and
        # raises if it ever is not -- and makes the transverse particle count
        # independent of resolution, so the total grows linearly in `nx` rather
        # than quadratically.
        transverseSpacings=80,
        # Set False to sample both states on the same lattice instead, leaving
        # the dense side's particles 4x heavier. The sampling cell below prints
        # what that costs before you run anything.
        equalMass=True,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`sod2dCase.buildSystem` -> `buildSodND`), not re-derived here. The build
# reports the lattice it settled on, and replaces the domain's transverse
# bounds with the slab it snapped to.
ctx = buildContext(sod2dCase, spec)
sod2dCase.configureScheme(ctx)
system = sod2dCase.buildSystem(ctx)
runningState = system.initializeNewState()

left, right = states(ctx)
print(f'\ndomain {ctx.config.domain.min.tolist()} .. {ctx.config.domain.max.tolist()}')
print(f'{runningState.state.positions.shape[0]} particles, dt = {float(ctx.config.dt):.4g}')

## What the sampler picked, and what the alternative costs

`sodSampling` is pure arithmetic -- no particles, no kernels -- so the two
integers it has to choose for the light side can be inspected on their own.
In 2D both usually come out exact; 3D (`sod_3d.ipynb`) is where it gets
interesting, since $4^{1/3}$ is irrational and no pair of commensurate
periodic lattices can match masses exactly there.

In [ ]:
for equalMass in (True, False):
    sampling = sodSampling(spec.nx, spec.dim, spec.L, spec.param('transverseSpacings'),
                           left, right, equalMass=equalMass)
    print(f'equalMass={equalMass}:')
    print(sodSamplingReport(sampling))

print('\nacross resolutions (equalMass=True):')
for nx in (25, 50, 100, 200, 400):
    sampling = sodSampling(nx, spec.dim, spec.L, spec.param('transverseSpacings'), left, right)
    count = (sampling.dense[0] * sampling.dense[1] ** (spec.dim - 1)
             + sampling.light[0] * sampling.light[1] ** (spec.dim - 1))
    print(f'  nx={nx:>4d}: {count:>7d} particles, mass mismatch '
          f'{abs(sampling.massRatio - 1):>7.3%}, worst cell aspect {sampling.anisotropy:.4f}')

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme,
                            ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# `scatter=True` from the first frame on, unlike the 1D notebook: many
# particles share an x here, so a line through them would be meaningless.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    fig, axis = plotSod(runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                        ctx.param('gamma'), left, right,
                        plotReference=True, plotLabels=False, scatter=True, t_=runningState.t)
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = sod2dCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData,
                              extraFields=sod2dCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation, an extra diagnostic, or a gradient step can be injected
# directly around it (`sod_backprop.ipynb` does the latter, in 1D).
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt))

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point -------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -----------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = sod2dCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        for ax in axis.flatten():
            ax.clear()
        plotSod_(fig, axis, runningState.state, ctx.config, ctx.schemeConfig, ctx.config.domain,
                 ctx.param('gamma'), left, right,
                 plotReference=True, plotLabels=False, scatter=True, t_=runningState.t)
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                   schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                   extraFields=sod2dCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## The slab itself

The profile panels collapse $y$ away on purpose. This is the same density
field left in place, drawn with the particle visualizer the 2D examples use
(`warpSPHPlotting.visualize`, through the runner's
backend-with-fallback helper). The coarser light-side lattice is visible
directly, and so is the fact that the waves stay flat in $y$ -- a bowed
contact or a ragged shock front here is the same error the profile scatter
reports as vertical spread, but located.

The full domain is 10:1 and draws as a sliver, so `xWindow` selects a piece
of it; the default is the half the analytic solution describes, and narrowing
it to the contact and the shock (`(0.4, 0.8)` at $t = 0.15$) is where there is
anything to see.

In [ ]:
from warpSPHPlotting import PlottingOptions, UniformColorMap   # noqa: E402

xWindow = (0.0, 1.0)          # try (0.4, 0.8) to zoom on the contact and the shock


def windowed(state, domain, xWindow):
    """The particles inside an x window, with a domain box to match, ready for
    `visualize`."""
    keep = ((state.positions[:, 0] >= xWindow[0]) & (state.positions[:, 0] <= xWindow[1]))
    perParticle = {f.name: getattr(state, f.name)[keep]
                   for f in dataclasses.fields(state)
                   if torch.is_tensor(getattr(state, f.name))
                   and getattr(state, f.name).shape[:1] == state.positions.shape[:1]}
    low, high = domain.min.clone(), domain.max.clone()
    low[0], high[0] = xWindow
    return (dataclasses.replace(state, **perParticle),
            type(domain)(low, high, domain.periodic, domain.dim))


slabState, slabDomain = windowed(runningState.state, ctx.config.domain, xWindow)

plotter = visualizeWithFallback(
    ctx, 'matplotlib',        # a few thousand particles; vispy is for 10^5
    particleState=slabState,
    domain=slabDomain,
    quantities={'A': slabState.densities},
    # A uniform map, not the diverging one the 2D examples reach for: density
    # here runs monotonically from the light state to the dense one and has no
    # meaningful midpoint to diverge about.
    plotOptions={'A': PlottingOptions(
        colorMap=UniformColorMap.viridis, markerSize=6,
        plotTitle='density', vMin=0.2, vMax=1.05)},
    figTitle=f'{sod2dCase.name}  t = {float(runningState.t):.4g}  '
             f'({slabState.positions.shape[0]} of '
             f'{runningState.state.positions.shape[0]} particles)',
    mosaic='A', figsize=(11, 3),
)
plotter.export(os.path.join(ctx.imagePath, 'slab.png'), dpi=150)
# figureOf(plotter)